## Importing Libraries

In [41]:
# imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn import set_config
import joblib

## Data abstraction from files

In [21]:
df = pd.read_csv('../Dataset/Titanic.csv')
df

,Unnamed: 0,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [22]:
df.drop(columns = ['class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone'], inplace = True)

In [23]:
df = df.iloc[:,1:]
df.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked
227,0,3,male,20.5,0,0,7.250,S
88,1,1,female,23.0,3,2,263.000,S
70,0,2,male,32.0,0,0,10.500,S
380,1,1,female,42.0,0,0,227.525,C
278,0,3,male,7.0,4,1,29.125,Q


In [24]:
df.isnull().sum()

survived      0
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
dtype: int64

## Spliting datas

In [25]:
x_train, x_test, y_train, y_test = train_test_split(df.drop(columns = ['survived']), df['survived'], test_size = 0.2, random_state = 45)

In [32]:
numeric_features = ['age', 'sibsp', 'parch', 'fare', 'pclass']
categorical_features = ['sex', 'embarked']

In [33]:
# Numerical
num_imputer = SimpleImputer(strategy='median')
num_scaler = MinMaxScaler()

# Categorical
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_encoder = OneHotEncoder(handle_unknown='ignore')

In [34]:
num_pipeline = Pipeline([
    ('imputer', num_imputer),
    ('scaler', num_scaler)
])

cat_pipeline = Pipeline([
    ('imputer', cat_imputer),
    ('encoder', cat_encoder)
])

In [35]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, numeric_features),
    ('cat', cat_pipeline, categorical_features)
])

In [36]:
feature_selector = SelectKBest(score_func=chi2, k=5)
model = DecisionTreeClassifier(random_state=42)

In [37]:
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('feature_selection', feature_selector),
    ('model', model)
])

In [38]:
set_config(display='diagram')
pipe

,steps,"[('preprocessor', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [39]:
pipe.fit(x_train, y_train)

,steps,"[('preprocessor', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [42]:
joblib.dump(pipe, "titanic_pipeline.pkl")

['titanic_pipeline.pkl']